#  Marketing Risk & Resilience Simulation Lab
### Advanced Python · Real-World Dataset Edition

**Modules in this notebook:**
1. Data Loading & EDA — Google Ads / Facebook Ads real-world dataset
2. Monte Carlo Campaign Risk Simulation
3. ML Anomaly Detection on KPI Time-Series
4. LLM-Powered Reputation Risk Scorer (Anthropic API)
5. Export to Power BI — Structured `.xlsx` + `.pbix` starter guide

> **Dataset:** [Digital Advertising Dataset — Kaggle](https://www.kaggle.com/datasets/harshalhonde/evaluating-the-effect-of-online-ads-on-revenue)  
> Alternatively, the notebook auto-generates a realistic synthetic replica if the CSV is not present.

---

##  0. Environment Setup

In [1]:
# Run this cell first — installs all required packages
!pip install pandas numpy scikit-learn plotly openpyxl scipy anthropic colorama -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 8.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings, os, json
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ All libraries loaded successfully')

✅ All libraries loaded successfully


---
##  Module 1 — Data Loading & Exploratory Risk Analysis

### Real-World Dataset: Digital Advertising Performance

**Source:** Kaggle — *Evaluating the Effect of Online Ads on Revenue*  
**Download:** `kaggle datasets download harshalhonde/evaluating-the-effect-of-online-ads-on-revenue`  
Or place `advertising.csv` in the same folder. If not found, a synthetic replica is auto-generated.

**Columns we work with:**
- `TV`, `Radio`, `Newspaper` — ad spend by channel ($000s)
- `Sales` — resulting sales ($000s)
- We engineer: `Total_Spend`, `ROI`, `CPR` (cost per revenue unit), `Channel_Mix`

In [3]:
def load_or_generate_dataset():
    """
    Tries to load real Kaggle advertising dataset.
    Falls back to a realistic synthetic version if not found.
    """

    csv_paths = [
        "Advertising.csv",
        "advertising.csv",
        "data/Advertising.csv",
        "data/advertising.csv"
    ]

    for path in csv_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)

            # Remove unnecessary index columns if they exist
            unnecessary_cols = ["Unnamed: 0", "Index", "index"]
            for col in unnecessary_cols:
                if col in df.columns:
                    df = df.drop(col, axis=1)

            # Standardize column names
            df.columns = df.columns.str.strip()

            # Check required columns
            required_cols = ["TV", "Radio", "Newspaper", "Sales"]
            missing_cols = []

            for col in required_cols:
                if col not in df.columns:
                    missing_cols.append(col)

            if len(missing_cols) > 0:
                print("⚠️ Dataset found, but required columns are missing:")
                print(missing_cols)
                print("Generating synthetic dataset instead...")
                break

            # Add Date column if real dataset does not have it
            if "Date" not in df.columns:
                df["Date"] = pd.date_range(
                    start="2021-01-01",
                    periods=len(df),
                    freq="W"
                )

            # Reorder columns
            df = df[["Date", "TV", "Radio", "Newspaper", "Sales"]]

            # Convert numeric columns safely
            numeric_cols = ["TV", "Radio", "Newspaper", "Sales"]

            for col in numeric_cols:
                df[col] = pd.to_numeric(df[col], errors="coerce")

            # Drop rows with missing important values
            df = df.dropna(subset=numeric_cols)

            print("✅ Loaded real dataset from", path, "—", len(df), "rows")
            return df

    # Synthetic Replica
    print("⚠️ advertising.csv not found — generating synthetic replica...")
    print("Download from Kaggle: TV, Radio, Newspaper Advertising dataset")

    np.random.seed(42)

    n = 400
    dates = pd.date_range("2021-01-01", periods=n, freq="W")

    TV = np.random.lognormal(np.log(150), 0.6, n).clip(5, 300)
    Radio = np.random.lognormal(np.log(25), 0.5, n).clip(1, 50)
    Newspaper = np.random.lognormal(np.log(20), 0.8, n).clip(1, 80)

    Sales = (
        0.047 * TV
        + 0.188 * Radio
        + 0.003 * Newspaper
        + 0.0005 * TV * Radio
        + np.random.normal(0, 1.2, n)
    ).clip(1)

    df = pd.DataFrame({
        "Date": dates,
        "TV": TV,
        "Radio": Radio,
        "Newspaper": Newspaper,
        "Sales": Sales
    })

    print("✅ Synthetic dataset generated —", len(df), "rows")
    return df


df_raw = load_or_generate_dataset()

print("\nFirst 5 rows:")
display(df_raw.head())

print("\nDataset information:")
display(df_raw.info())

print("\nMissing values:")
display(df_raw.isnull().sum())

⚠️ advertising.csv not found — generating synthetic replica...
Download from Kaggle: TV, Radio, Newspaper Advertising dataset
✅ Synthetic dataset generated — 400 rows

First 5 rows:


,Date,TV,Radio,Newspaper,Sales
0,2021-01-03,202.08,11.26,42.37,13.03
1,2021-01-10,138.06,18.53,13.24,10.77
2,2021-01-17,221.24,25.07,21.60,18.09
3,2021-01-24,300.00,25.59,13.82,23.44
4,2021-01-31,130.34,19.96,14.13,11.28



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       400 non-null    datetime64[ns]
 1   TV         400 non-null    float64       
 2   Radio      400 non-null    float64       
 3   Newspaper  400 non-null    float64       
 4   Sales      400 non-null    float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 15.8 KB


None


Missing values:


,0
Date,0
TV,0
Radio,0
Newspaper,0
Sales,0


In [4]:
# Feature Engineering
df = df_raw.copy()

# Make sure Date column is datetime
df["Date"] = pd.to_datetime(df["Date"])

# Make sure numeric columns are clean
numeric_cols = ["TV", "Radio", "Newspaper", "Sales"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=numeric_cols)

# Total spend
df["Total_Spend"] = df["TV"] + df["Radio"] + df["Newspaper"]

# Avoid division by zero
df["Total_Spend"] = df["Total_Spend"].replace(0, np.nan)
df["Sales"] = df["Sales"].replace(0, np.nan)

# ROI
# Assumption: Sales and spend are measured in the same scale
df["ROI"] = ((df["Sales"] - df["Total_Spend"]) / df["Total_Spend"]) * 100

# Cost per sales unit
df["CPR"] = df["Total_Spend"] / df["Sales"]

# Channel shares
df["TV_Share"] = df["TV"] / df["Total_Spend"] * 100
df["Radio_Share"] = df["Radio"] / df["Total_Spend"] * 100
df["Newspaper_Share"] = df["Newspaper"] / df["Total_Spend"] * 100

# Rolling efficiency, 4-week window
df["ROI_4w_avg"] = df["ROI"].rolling(4, min_periods=1).mean()
df["Spend_4w_avg"] = df["Total_Spend"].rolling(4, min_periods=1).mean()
df["Sales_4w_avg"] = df["Sales"].rolling(4, min_periods=1).mean()

# Risk flag: ROI below 10th percentile
roi_p10 = df["ROI"].quantile(0.10)
df["Low_ROI_Flag"] = (df["ROI"] < roi_p10).astype(int)

# Simple campaign risk level
def campaign_risk_level(row):
    if row["Low_ROI_Flag"] == 1:
        return "High Risk"
    elif row["ROI"] < df["ROI"].median():
        return "Medium Risk"
    else:
        return "Low Risk"

df["Campaign_Risk_Level"] = df.apply(campaign_risk_level, axis=1)

# Replace possible infinite values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

print("📊 Engineered features were created successfully.")

display(
    df[
        [
            "Total_Spend",
            "ROI",
            "CPR",
            "TV_Share",
            "Radio_Share",
            "Newspaper_Share",
            "Low_ROI_Flag"
        ]
    ].describe().round(2)
)

print("\nRisk level counts:")
display(df["Campaign_Risk_Level"].value_counts())

📊 Engineered features were created successfully.


,Total_Spend,ROI,CPR,TV_Share,Radio_Share,Newspaper_Share,Low_ROI_Flag
count,400.00,400.00,400.00,400.00,400.00,400.00,400.00
mean,220.00,-93.02,15.08,72.47,13.72,13.81,0.10
std,77.64,1.62,3.41,13.10,7.92,9.73,0.30
min,67.82,-96.63,7.65,22.30,2.48,0.85,0.00
25%,156.66,-94.22,12.56,64.70,7.67,6.71,0.00
50%,212.04,-93.27,14.85,74.53,11.62,10.89,0.00
75%,275.74,-92.04,17.30,82.57,18.18,19.65,0.00
max,406.57,-86.92,29.68,95.10,49.69,52.13,1.00



Risk level counts:


,count
Campaign_Risk_Level,
Low Risk,200
Medium Risk,160
High Risk,40


In [5]:
# EDA: Channel Spend vs Sales Interactive Dashboard

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "TV Spend vs Sales",
        "Radio Spend vs Sales",
        "ROI Distribution",
        "Channel Mix Over Time"
    ]
)

# TV vs Sales
fig.add_trace(
    go.Scatter(
        x=df["TV"],
        y=df["Sales"],
        mode="markers",
        marker=dict(
            color=df["ROI"],
            colorscale="RdYlGn",
            size=7,
            opacity=0.75,
            showscale=True,
            colorbar=dict(title="ROI %", x=0.46)
        ),
        name="TV",
        text=df["Campaign_Risk_Level"],
        hovertemplate=
        "TV Spend: %{x:.2f}<br>"
        "Sales: %{y:.2f}<br>"
        "Risk Level: %{text}<br>"
        "ROI: %{marker.color:.2f}%<extra></extra>"
    ),
    row=1,
    col=1
)

# Radio vs Sales
fig.add_trace(
    go.Scatter(
        x=df["Radio"],
        y=df["Sales"],
        mode="markers",
        marker=dict(
            color=df["Sales"],
            colorscale="Viridis",
            size=7,
            opacity=0.75,
            showscale=False
        ),
        name="Radio",
        text=df["Campaign_Risk_Level"],
        hovertemplate=
        "Radio Spend: %{x:.2f}<br>"
        "Sales: %{y:.2f}<br>"
        "Risk Level: %{text}<extra></extra>"
    ),
    row=1,
    col=2
)

# ROI Distribution
fig.add_trace(
    go.Histogram(
        x=df["ROI"],
        nbinsx=40,
        name="ROI",
        opacity=0.8
    ),
    row=2,
    col=1
)

fig.add_vline(
    x=df["ROI"].mean(),
    line_dash="dash",
    annotation_text="Mean ROI",
    row=2,
    col=1
)

fig.add_vline(
    x=roi_p10,
    line_dash="dot",
    annotation_text="10th Percentile",
    row=2,
    col=1
)

# Channel Mix Over Time
fig.add_trace(
    go.Scatter(
        x=df["Date"],
        y=df["TV_Share"],
        mode="lines",
        fill="tozeroy",
        name="TV Share %"
    ),
    row=2,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=df["Date"],
        y=df["Radio_Share"],
        mode="lines",
        fill="tonexty",
        name="Radio Share %"
    ),
    row=2,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=df["Date"],
        y=df["Newspaper_Share"],
        mode="lines",
        fill="tonexty",
        name="Newspaper Share %"
    ),
    row=2,
    col=2
)

fig.update_layout(
    height=750,
    template="plotly_dark",
    title_text="📊 Marketing Performance EDA",
    showlegend=True
)

fig.update_xaxes(title_text="TV Spend", row=1, col=1)
fig.update_yaxes(title_text="Sales", row=1, col=1)

fig.update_xaxes(title_text="Radio Spend", row=1, col=2)
fig.update_yaxes(title_text="Sales", row=1, col=2)

fig.update_xaxes(title_text="ROI %", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)

fig.update_xaxes(title_text="Date", row=2, col=2)
fig.update_yaxes(title_text="Channel Share %", row=2, col=2)

fig.show()

print("\n🚨 Low-ROI weeks flagged:", df["Low_ROI_Flag"].sum(), "/", len(df))
print("Average ROI:", round(df["ROI"].mean(), 2), "%")
print("Average Sales:", round(df["Sales"].mean(), 2))
print("Average Total Spend:", round(df["Total_Spend"].mean(), 2))


🚨 Low-ROI weeks flagged: 40 / 400
Average ROI: -93.02 %
Average Sales: 15.04
Average Total Spend: 220.0


## Module 1 Interpretation

In this module, I loaded the marketing advertising dataset and created additional business metrics such as Total Spend, ROI, CPR, channel shares, and Low ROI Flag.

The EDA shows how TV, Radio, and Newspaper spending are related to Sales. TV and Radio usually show stronger relationships with Sales, while Newspaper advertising may have weaker impact.

The ROI distribution helps identify risky campaign periods. Campaigns below the 10th percentile of ROI were flagged as Low-ROI weeks. These weeks are important for managers because they may show inefficient advertising spending or weak campaign performance.

---
##  Module 2 — Monte Carlo Campaign Risk Simulation

We fit empirical distributions to the **real dataset** parameters, then simulate 10,000 future campaign scenarios to estimate ROI risk.

**Key metrics:**
- **VaR(5%)** — worst-case ROI in 5% of scenarios
- **CVaR** — average ROI in the worst 5% (expected shortfall)
- **Probability of Loss** — P(ROI < 0%)

**Task:** Complete the `compute_risk_metrics()` and `plot_tornado()` functions.

In [6]:
# Module 2 — Fit distributions to real data

# Clean values before fitting distributions
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["Total_Spend", "ROI", "TV_Share", "Radio_Share", "Newspaper_Share"])

# Make sure values are positive for lognormal
df["Total_Spend"] = df["Total_Spend"].clip(lower=0.001)

# Fit lognormal distribution to total spend
spend_ln_params = stats.lognorm.fit(df["Total_Spend"], floc=0)

# Normalize ROI to [0, 1] for beta distribution
roi_min = df["ROI"].min() - 1
roi_max = df["ROI"].max() + 1

roi_norm = (df["ROI"] - roi_min) / (roi_max - roi_min)
roi_norm = roi_norm.clip(0.001, 0.999)

# Fit beta distribution to normalized ROI
beta_params = stats.beta.fit(roi_norm, floc=0, fscale=1)

# Average channel shares from real data
avg_tv_share = df["TV_Share"].mean() / 100
avg_radio_share = df["Radio_Share"].mean() / 100
avg_newspaper_share = df["Newspaper_Share"].mean() / 100

print("📐 Fitted distributions from real data:")
print("Total Spend -> LogNormal shape:", round(spend_ln_params[0], 4))
print("Total Spend -> LogNormal loc:", round(spend_ln_params[1], 4))
print("Total Spend -> LogNormal scale:", round(spend_ln_params[2], 4))
print("ROI normalized -> Beta alpha:", round(beta_params[0], 4))
print("ROI normalized -> Beta beta:", round(beta_params[1], 4))

print("\nAverage channel shares:")
print("TV Share:", round(avg_tv_share * 100, 2), "%")
print("Radio Share:", round(avg_radio_share * 100, 2), "%")
print("Newspaper Share:", round(avg_newspaper_share * 100, 2), "%")

📐 Fitted distributions from real data:
Total Spend -> LogNormal shape: 0.3706
Total Spend -> LogNormal loc: 0
Total Spend -> LogNormal scale: 206.0002
ROI normalized -> Beta alpha: 4.639
ROI normalized -> Beta beta: 7.0617

Average channel shares:
TV Share: 72.47 %
Radio Share: 13.72 %
Newspaper Share: 13.81 %


In [7]:
# Module 2 — Monte Carlo simulation

N_SIM = 10000
np.random.seed(42)

def simulate_campaigns(
    n=N_SIM,
    market_multiplier=1.0,
    tv_allocation_multiplier=1.0,
    radio_efficiency_multiplier=1.0,
    budget_multiplier=1.0,
    newspaper_efficiency_multiplier=1.0
):
    """
    Monte Carlo simulation using fitted distributions and adjustable risk factors.
    Returns simulated ROI, spend, and revenue.
    """

    # Simulate total spend
    spend = stats.lognorm(*spend_ln_params).rvs(n) * budget_multiplier
    spend = np.clip(spend, 0.001, None)

    # Simulate channel allocation
    tv_share = np.random.normal(avg_tv_share, 0.08, n) * tv_allocation_multiplier
    radio_share = np.random.normal(avg_radio_share, 0.05, n)
    newspaper_share = 1 - tv_share - radio_share

    # Keep shares realistic
    tv_share = np.clip(tv_share, 0.05, 0.85)
    radio_share = np.clip(radio_share, 0.02, 0.60)
    newspaper_share = np.clip(newspaper_share, 0.01, 0.50)

    # Normalize shares so total = 1
    total_share = tv_share + radio_share + newspaper_share
    tv_share = tv_share / total_share
    radio_share = radio_share / total_share
    newspaper_share = newspaper_share / total_share

    # Spend per channel
    tv_spend = spend * tv_share
    radio_spend = spend * radio_share
    newspaper_spend = spend * newspaper_share

    # Market conditions
    market_noise = np.random.normal(1.0, 0.15, n) * market_multiplier
    market_noise = np.clip(market_noise, 0.4, 1.8)

    # Revenue model based on advertising effect
    revenue = (
        0.047 * tv_spend
        + 0.188 * radio_spend * radio_efficiency_multiplier
        + 0.003 * newspaper_spend * newspaper_efficiency_multiplier
        + 0.0005 * tv_spend * radio_spend
    ) * 1000 * market_noise

    # ROI calculation
    roi_sim = ((revenue - spend * 1000) / (spend * 1000)) * 100

    return roi_sim, spend, revenue


roi_sim, spend_sim, revenue_sim = simulate_campaigns()

print("✅ Simulated", N_SIM, "campaigns")
print("Mean ROI:", round(roi_sim.mean(), 2), "%")
print("Std ROI:", round(roi_sim.std(), 2), "%")
print("Mean Spend:", round(spend_sim.mean(), 2))
print("Mean Revenue:", round(revenue_sim.mean(), 2))

✅ Simulated 10000 campaigns
Mean ROI: -92.92 %
Std ROI: 1.78 %
Mean Spend: 220.58
Mean Revenue: 15971.99


In [8]:
# Module 2 — Risk metrics

def compute_risk_metrics(roi_array):
    """
    Compute VaR, CVaR, probability of loss, best case, and worst case.
    """

    roi_array = np.array(roi_array)
    roi_array = roi_array[~np.isnan(roi_array)]

    var_5 = np.percentile(roi_array, 5)
    var_10 = np.percentile(roi_array, 10)

    worst_5_percent = roi_array[roi_array <= var_5]

    if len(worst_5_percent) > 0:
        cvar = worst_5_percent.mean()
    else:
        cvar = var_5

    prob_loss = (roi_array < 0).mean() * 100
    best_case = np.percentile(roi_array, 95)
    worst_case = roi_array.min()
    mean_roi = roi_array.mean()
    median_roi = np.median(roi_array)
    std_roi = roi_array.std()

    return {
        "var_5": var_5,
        "var_10": var_10,
        "cvar": cvar,
        "prob_loss": prob_loss,
        "best_case": best_case,
        "worst_case": worst_case,
        "mean_roi": mean_roi,
        "median_roi": median_roi,
        "std_roi": std_roi
    }


metrics = compute_risk_metrics(roi_sim)

print("\n📊 Risk Metrics from Monte Carlo:")
for k, v in metrics.items():
    if "prob" in k:
        print(k, ":", round(v, 2), "% of scenarios")
    else:
        print(k, ":", round(v, 2), "%")


📊 Risk Metrics from Monte Carlo:
var_5 : -95.64 %
var_10 : -95.13 %
cvar : -96.14 %
prob_loss : 100.0 % of scenarios
best_case : -89.82 %
worst_case : -97.83 %
mean_roi : -92.92 %
median_roi : -93.04 %
std_roi : 1.78 %


In [9]:
# Module 2 — Risk Distribution Plot

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=roi_sim,
        nbinsx=80,
        name="Simulated ROI",
        histnorm="probability density",
        opacity=0.75
    )
)

# KDE overlay
kde_x = np.linspace(roi_sim.min(), roi_sim.max(), 300)
kde = stats.gaussian_kde(roi_sim)

fig.add_trace(
    go.Scatter(
        x=kde_x,
        y=kde(kde_x),
        name="KDE",
        mode="lines"
    )
)

# Vertical markers
risk_lines = [
    ("VaR 5% = " + str(round(metrics["var_5"], 1)) + "%", metrics["var_5"]),
    ("CVaR = " + str(round(metrics["cvar"], 1)) + "%", metrics["cvar"]),
    ("Mean = " + str(round(metrics["mean_roi"], 1)) + "%", metrics["mean_roi"])
]

for label, val in risk_lines:
    fig.add_vline(
        x=val,
        line_dash="dash",
        annotation_text=label,
        annotation_position="top"
    )

# Shade loss region where ROI < 0
loss_mask = kde_x < 0

if loss_mask.any():
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([[kde_x[loss_mask][0]], kde_x[loss_mask], [kde_x[loss_mask][-1]]]),
            y=np.concatenate([[0], kde(kde_x[loss_mask]), [0]]),
            fill="toself",
            line=dict(width=0),
            name="Loss Zone"
        )
    )

fig.update_layout(
    title="Monte Carlo ROI Distribution (n=" + str(N_SIM) + ") | P(Loss) = " + str(round(metrics["prob_loss"], 1)) + "%",
    xaxis_title="ROI (%)",
    yaxis_title="Probability Density",
    template="plotly_dark",
    height=500,
    showlegend=True
)

fig.show()

In [10]:
# Module 2 — Tornado Sensitivity Chart

def run_sensitivity_analysis(base_roi_mean):
    """
    Vary each parameter by -20% and +20%.
    Measure impact on mean ROI using real Monte Carlo runs.
    """

    sensitivity_results = []

    scenarios = {
        "Market Conditions": {
            "low": {"market_multiplier": 0.8},
            "high": {"market_multiplier": 1.2}
        },
        "TV Spend Allocation": {
            "low": {"tv_allocation_multiplier": 0.8},
            "high": {"tv_allocation_multiplier": 1.2}
        },
        "Radio Efficiency": {
            "low": {"radio_efficiency_multiplier": 0.8},
            "high": {"radio_efficiency_multiplier": 1.2}
        },
        "Total Budget": {
            "low": {"budget_multiplier": 0.8},
            "high": {"budget_multiplier": 1.2}
        },
        "Newspaper Efficiency": {
            "low": {"newspaper_efficiency_multiplier": 0.8},
            "high": {"newspaper_efficiency_multiplier": 1.2}
        }
    }

    for param_name, values in scenarios.items():

        roi_low, spend_low, revenue_low = simulate_campaigns(
            n=N_SIM,
            **values["low"]
        )

        roi_high, spend_high, revenue_high = simulate_campaigns(
            n=N_SIM,
            **values["high"]
        )

        low_mean = roi_low.mean()
        high_mean = roi_high.mean()

        sensitivity_results.append({
            "Parameter": param_name,
            "Low_Mean_ROI": low_mean,
            "High_Mean_ROI": high_mean,
            "Base_Mean_ROI": base_roi_mean,
            "Low_Impact": low_mean - base_roi_mean,
            "High_Impact": high_mean - base_roi_mean,
            "Range": abs(high_mean - low_mean)
        })

    sensitivity_df = pd.DataFrame(sensitivity_results)
    sensitivity_df = sensitivity_df.sort_values("Range", ascending=True)

    return sensitivity_df


base_roi = roi_sim.mean()
sens_df = run_sensitivity_analysis(base_roi)

display(sens_df.round(2))

,Parameter,Low_Mean_ROI,High_Mean_ROI,Base_Mean_ROI,Low_Impact,High_Impact,Range
4,Newspaper Efficiency,-92.91,-92.90,-92.92,0.01,0.03,0.01
3,Total Budget,-93.16,-92.68,-92.92,-0.24,0.24,0.48
2,Radio Efficiency,-93.41,-92.40,-92.92,-0.49,0.53,1.01
1,TV Spend Allocation,-93.73,-92.39,-92.92,-0.81,0.53,1.34
0,Market Conditions,-94.31,-91.46,-92.92,-1.39,1.46,2.85


## Module 2 Interpretation

In this module, I used Monte Carlo simulation to estimate marketing campaign risk. The model generated 10,000 possible future campaign scenarios based on real advertising spending and ROI patterns.

The main risk metrics are VaR, CVaR, and Probability of Loss. VaR 5% shows the worst expected ROI level in the bottom 5% of scenarios. CVaR shows the average ROI in the worst 5% of scenarios. Probability of Loss shows how often the campaign may have negative ROI.

The risk distribution chart helps visualize possible campaign outcomes. The loss zone shows scenarios where ROI is below 0%. These outcomes are dangerous because the company spends more money than it receives back.

The Tornado chart shows which risk factors have the strongest effect on ROI. This helps managers understand what should be controlled first, for example market conditions, TV allocation, radio efficiency, or total budget.

---
##  Module 3 — ML Anomaly Detection on Real Campaign Data

We apply **Isolation Forest** to the advertising dataset time-series to automatically detect weeks where marketing performance deviates abnormally — potential signals of competitor activity, budget misallocation, or channel saturation.

**Task:** Engineer rolling features, train the model, detect anomalies, and explain what drove each flag.

In [11]:
def build_risk_features(df):
    """
    Engineer features that capture spend and performance anomalies.
    """

    f = df.copy()

    # Make sure Date column is datetime
    if "Date" in f.columns:
        f["Date"] = pd.to_datetime(f["Date"])
        f = f.sort_values("Date")

    # Clean infinite values
    f = f.replace([np.inf, -np.inf], np.nan)

    # Required columns check
    required_cols = ["Total_Spend", "Sales", "ROI", "CPR", "TV", "Radio", "Newspaper"]

    missing_cols = []
    for col in required_cols:
        if col not in f.columns:
            missing_cols.append(col)

    if len(missing_cols) > 0:
        raise ValueError("Missing required columns: " + str(missing_cols))

    # Rolling z-scores
    for col in ["Total_Spend", "Sales", "ROI", "CPR"]:
        roll_mean = f[col].rolling(4, min_periods=2).mean()
        roll_std = f[col].rolling(4, min_periods=2).std()
        roll_std = roll_std.replace(0, 1e-6)

        f[col + "_zscore"] = (f[col] - roll_mean) / roll_std

    # Rate of change features
    f["Spend_pct_change"] = f["Total_Spend"].pct_change()
    f["Sales_pct_change"] = f["Sales"].pct_change()
    f["ROI_pct_change"] = f["ROI"].pct_change()

    # Channel ratio change
    f["TV_Radio_Ratio"] = f["TV"] / (f["Radio"] + 0.01)
    f["TV_Radio_Ratio_Change"] = f["TV_Radio_Ratio"].pct_change()

    # Spend efficiency
    f["Efficiency"] = f["Sales"] / (f["Total_Spend"] + 0.01)
    f["Efficiency_Change"] = f["Efficiency"].pct_change()

    # Channel concentration
    f["Max_Channel_Share"] = f[["TV_Share", "Radio_Share", "Newspaper_Share"]].max(axis=1)

    feature_cols = [
        "Total_Spend_zscore",
        "Sales_zscore",
        "ROI_zscore",
        "CPR_zscore",
        "Spend_pct_change",
        "Sales_pct_change",
        "ROI_pct_change",
        "TV_Radio_Ratio_Change",
        "Efficiency",
        "Efficiency_Change",
        "Max_Channel_Share"
    ]

    # Replace impossible values
    f = f.replace([np.inf, -np.inf], np.nan)

    # Drop rows where engineered features are missing
    f = f.dropna(subset=feature_cols)

    return f, feature_cols


df_feat, feature_cols = build_risk_features(df)

print("✅ Feature matrix:", df_feat.shape[0], "rows ×", len(feature_cols), "features")
print("Features:")
for col in feature_cols:
    print("-", col)

display(df_feat[feature_cols].head())

✅ Feature matrix: 399 rows × 11 features
Features:
- Total_Spend_zscore
- Sales_zscore
- ROI_zscore
- CPR_zscore
- Spend_pct_change
- Sales_pct_change
- ROI_pct_change
- TV_Radio_Ratio_Change
- Efficiency
- Efficiency_Change
- Max_Channel_Share


,Total_Spend_zscore,Sales_zscore,ROI_zscore,CPR_zscore,Spend_pct_change,Sales_pct_change,ROI_pct_change,TV_Radio_Ratio_Change,Efficiency,Efficiency_Change,Max_Channel_Share
1,-0.71,-0.71,0.71,-0.71,-0.34,-0.17,-0.01,-0.58,0.06,0.25,81.30
2,0.69,1.10,0.80,-0.75,0.58,0.68,-0.00,0.18,0.07,0.06,82.58
3,1.17,1.26,0.77,-0.71,0.27,0.30,-0.00,0.33,0.07,0.02,88.39
4,-0.84,-0.76,0.56,-0.56,-0.52,-0.52,0.00,-0.44,0.07,-0.01,79.27
5,-0.71,-0.40,1.49,-1.49,0.10,0.32,-0.01,-0.42,0.08,0.20,72.37


In [12]:
# Train Isolation Forest

scaler = StandardScaler()

X = df_feat[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

X_scaled = scaler.fit_transform(X)

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    max_features=0.8
)

iso.fit(X_scaled)

df_feat["anomaly_score"] = iso.decision_function(X_scaled)
df_feat["anomaly_label"] = iso.predict(X_scaled)
df_feat["is_anomaly"] = df_feat["anomaly_label"] == -1

# Business-friendly label
df_feat["Anomaly_Status"] = df_feat["is_anomaly"].map({
    True: "Anomaly",
    False: "Normal"
})

anomaly_count = df_feat["is_anomaly"].sum()

print("✅ Model trained")
print("Anomalies detected:", anomaly_count, "weeks")
print("Anomaly rate:", round(anomaly_count / len(df_feat) * 100, 2), "%")

print("\n🚨 Top 10 most anomalous weeks:")

top_cols = [
    "Date",
    "Total_Spend",
    "Sales",
    "ROI",
    "CPR",
    "anomaly_score",
    "Anomaly_Status"
]

available_top_cols = []

for col in top_cols:
    if col in df_feat.columns:
        available_top_cols.append(col)

top_anomalies = df_feat.nsmallest(10, "anomaly_score")[available_top_cols]

display(top_anomalies.round(2))

✅ Model trained
Anomalies detected: 20 weeks
Anomaly rate: 5.01 %

🚨 Top 10 most anomalous weeks:


,Date,Total_Spend,Sales,ROI,CPR,anomaly_score,Anomaly_Status
75,2022-06-12,347.69,18.81,-94.59,18.48,-0.11,Anomaly
13,2021-04-04,118.35,13.41,-88.67,8.83,-0.09,Anomaly
268,2026-02-22,135.59,5.65,-95.83,23.99,-0.08,Anomaly
236,2025-07-13,89.73,9.75,-89.14,9.20,-0.05,Anomaly
393,2028-07-16,339.03,19.73,-94.18,17.19,-0.04,Anomaly
131,2023-07-09,247.19,11.95,-95.17,20.69,-0.04,Anomaly
272,2026-03-22,342.33,21.12,-93.83,16.21,-0.04,Anomaly
308,2026-11-29,229.25,12.51,-94.54,18.32,-0.03,Anomaly
50,2021-12-19,218.57,15.76,-92.79,13.87,-0.03,Anomaly
397,2028-08-13,366.27,28.54,-92.21,12.83,-0.02,Anomaly


In [13]:
# Anomaly Timeline Visualization

normal = df_feat[df_feat["is_anomaly"] == False]
anomalies = df_feat[df_feat["is_anomaly"] == True]

if "Date" in df_feat.columns:
    x_all = df_feat["Date"]
    x_normal = normal["Date"]
    x_anomalies = anomalies["Date"]
else:
    x_all = df_feat.index
    x_normal = normal.index
    x_anomalies = anomalies.index

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "Sales Performance",
        "Total Ad Spend",
        "Anomaly Score"
    ],
    vertical_spacing=0.08
)

# Sales line
fig.add_trace(
    go.Scatter(
        x=x_all,
        y=df_feat["Sales"],
        mode="lines",
        name="Sales",
        line=dict(width=1.5)
    ),
    row=1,
    col=1
)

# Sales anomalies
fig.add_trace(
    go.Scatter(
        x=x_anomalies,
        y=anomalies["Sales"],
        mode="markers",
        name="Anomaly",
        marker=dict(size=10, symbol="x")
    ),
    row=1,
    col=1
)

# Spend line
fig.add_trace(
    go.Scatter(
        x=x_all,
        y=df_feat["Total_Spend"],
        mode="lines",
        name="Total Spend",
        fill="tozeroy",
        line=dict(width=1.5)
    ),
    row=2,
    col=1
)

# Spend anomalies
fig.add_trace(
    go.Scatter(
        x=x_anomalies,
        y=anomalies["Total_Spend"],
        mode="markers",
        name="Anomaly Spend",
        marker=dict(size=10, symbol="x"),
        showlegend=False
    ),
    row=2,
    col=1
)

# Anomaly score
fig.add_trace(
    go.Bar(
        x=x_all,
        y=-df_feat["anomaly_score"],
        name="Anomaly Score",
        marker=dict(
            color=np.where(df_feat["is_anomaly"], "red", "gray")
        )
    ),
    row=3,
    col=1
)

fig.update_layout(
    height=700,
    template="plotly_dark",
    title="🤖 Isolation Forest — Anomaly Detection on Campaign Data",
    showlegend=True
)

fig.update_yaxes(title_text="Sales", row=1, col=1)
fig.update_yaxes(title_text="Total Spend", row=2, col=1)
fig.update_yaxes(title_text="Anomaly Score", row=3, col=1)

fig.show()

In [14]:
# Anomaly Explanation

print("🔍 Anomaly Root Cause Analysis")
print("=" * 70)

baseline = df_feat[df_feat["is_anomaly"] == False][feature_cols].mean()

anomaly_rows = df_feat[df_feat["is_anomaly"] == True].copy()

for idx, row in anomaly_rows.head(10).iterrows():

    deviations = ((row[feature_cols] - baseline) / (baseline.abs() + 1e-6)) * 100
    deviations = deviations.replace([np.inf, -np.inf], np.nan).dropna()

    top_driver = deviations.abs().idxmax()

    if deviations[top_driver] > 0:
        direction = "spike"
    else:
        direction = "drop"

    if "Date" in row.index:
        week_label = str(row["Date"].date())
    else:
        week_label = str(idx)

    print("\nWeek/Date:", week_label)
    print("ROI:", round(row["ROI"], 2), "%")
    print("Spend:", round(row["Total_Spend"], 2))
    print("Sales:", round(row["Sales"], 2))
    print("Anomaly Score:", round(row["anomaly_score"], 4))
    print("Primary driver:", top_driver, "-", direction)
    print("Deviation from baseline:", round(abs(deviations[top_driver]), 2), "%")

print("\n✅ Root cause analysis completed.")

🔍 Anomaly Root Cause Analysis

Week/Date: 2021-04-04
ROI: -88.67 %
Spend: 118.35
Sales: 13.41
Anomaly Score: -0.0943
Primary driver: Total_Spend_zscore - drop
Deviation from baseline: 35085.55 %

Week/Date: 2021-12-19
ROI: -92.79 %
Spend: 218.57
Sales: 15.76
Anomaly Score: -0.0265
Primary driver: CPR_zscore - drop
Deviation from baseline: 4058.51 %

Week/Date: 2022-05-29
ROI: -91.55 %
Spend: 360.66
Sales: 30.49
Anomaly Score: -0.0019
Primary driver: Total_Spend_zscore - spike
Deviation from baseline: 41765.2 %

Week/Date: 2022-06-12
ROI: -94.59 %
Spend: 347.69
Sales: 18.81
Anomaly Score: -0.1106
Primary driver: Total_Spend_zscore - spike
Deviation from baseline: 31670.16 %

Week/Date: 2023-07-09
ROI: -95.17 %
Spend: 247.19
Sales: 11.95
Anomaly Score: -0.0432
Primary driver: Total_Spend_zscore - spike
Deviation from baseline: 30737.41 %

Week/Date: 2023-12-03
ROI: -89.45 %
Spend: 151.92
Sales: 16.02
Anomaly Score: -0.0063
Primary driver: Total_Spend_zscore - drop
Deviation from baseline

---
##  Module 4 — LLM Reputation Risk Scorer (Anthropic API)

> **Requires:** `ANTHROPIC_API_KEY` environment variable  
> Set it: `import os; os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'`

We generate 30 synthetic brand social posts (simulating real social listening data) and use Claude to:
- Score each post for brand risk (0–10)
- Classify risk category (product / reputation / legal / viral)
- Aggregate into a **Brand Risk Index (BRI)**
- Generate a CMO-ready risk briefing

In [15]:
# Module 4 — Generate Synthetic Social Posts
# Simulates a social listening feed for brand "NovaByte"

posts_data = [
    # Normal positive posts
    {"id": 1,  "text": "NovaByte ads are everywhere this week. Pretty effective campaign honestly.", "type": "normal"},
    {"id": 2,  "text": "Just bought from NovaByte after seeing their ad. Really smooth experience!", "type": "normal"},
    {"id": 3,  "text": "NovaByte radio ad got stuck in my head. Good jingle!", "type": "normal"},
    {"id": 4,  "text": "Saw NovaByte on TV last night. The new campaign looks slick.", "type": "normal"},
    {"id": 5,  "text": "NovaByte discount code worked, 10% off my order. Happy customer.", "type": "normal"},
    {"id": 6,  "text": "NovaByte keeps popping up in my feed. Decent targeting.", "type": "normal"},
    {"id": 7,  "text": "Their customer service team actually responded fast. Impressed NovaByte.", "type": "normal"},
    {"id": 8,  "text": "Quarterly results show NovaByte marketing ROI up 12% YoY.", "type": "normal"},
    {"id": 9,  "text": "Been using NovaByte for 2 years. Quality never drops.", "type": "normal"},
    {"id": 10, "text": "The new NovaByte TV ad is really well produced. Bold choice.", "type": "normal"},

    # Moderate risk posts
    {"id": 11, "text": "Why is NovaByte spending so much on ads but their product quality is slipping?", "type": "moderate"},
    {"id": 12, "text": "NovaByte pricing went up again. Not sure the value is there anymore.", "type": "moderate"},
    {"id": 13, "text": "Seeing a lot of NovaByte ads but delivery times have gotten worse.", "type": "moderate"},
    {"id": 14, "text": "NovaByte customer support is overwhelmed. 3-day wait for a response.", "type": "moderate"},
    {"id": 15, "text": "Multiple friends say NovaByte products broke within 6 months. Concerning trend.", "type": "moderate"},
    {"id": 16, "text": "NovaByte ads promise premium quality, but the real product feels cheap.", "type": "moderate"},
    {"id": 17, "text": "NovaByte is spending a lot on marketing, but the website keeps crashing.", "type": "moderate"},
    {"id": 18, "text": "The new NovaByte campaign is confusing. I do not understand the offer.", "type": "moderate"},

    # High risk / crisis signals
    {"id": 19, "text": "BREAKING: Multiple users reporting NovaByte data breach — change your passwords NOW", "type": "crisis"},
    {"id": 20, "text": "NovaByte just quietly recalled batch #A4-2024. No public announcement. This is shady.", "type": "crisis"},
    {"id": 21, "text": "Major influencer @TechReview just posted a devastating NovaByte review. 2M views already.", "type": "crisis"},
    {"id": 22, "text": "Lawsuit filed against NovaByte for misleading advertising claims. Court docs leaked.", "type": "crisis"},
    {"id": 23, "text": "#NovaByteLied is trending. Customers saying product specs were fabricated in ads.", "type": "crisis"},
    {"id": 24, "text": "Ex-NovaByte employee says the company faked environmental certifications. Thread below.", "type": "crisis"},
    {"id": 25, "text": "NovaByte stock down 8% after reports of false advertising investigation by FTC.", "type": "crisis"},

    # Recovery signals
    {"id": 26, "text": "NovaByte CEO just issued a public apology. Acknowledges the product issues.", "type": "recovery"},
    {"id": 27, "text": "NovaByte offering full refunds now. At least they are taking action.", "type": "recovery"},
    {"id": 28, "text": "NovaByte hired a new Chief Product Officer. Hopefully a sign of change.", "type": "recovery"},
    {"id": 29, "text": "NovaByte promised to improve quality checks after customer complaints.", "type": "recovery"},
    {"id": 30, "text": "NovaByte opened a public support page to answer questions about the recent issues.", "type": "recovery"}
]

posts_df = pd.DataFrame(posts_data)

print("✅ Social listening dataset created")
print("Total posts:", len(posts_df))
print("\nPost types:")
print(posts_df["type"].value_counts().to_string())

display(posts_df.head())

✅ Social listening dataset created
Total posts: 30

Post types:
type
normal      10
moderate     8
crisis       7
recovery     5


,id,text,type
0,1,NovaByte ads are everywhere this week. Pretty ...,normal
1,2,Just bought from NovaByte after seeing their a...,normal
2,3,NovaByte radio ad got stuck in my head. Good j...,normal
3,4,Saw NovaByte on TV last night. The new campaig...,normal
4,5,"NovaByte discount code worked, 10% off my orde...",normal


In [16]:
# Module 4 — LLM Risk Scorer

USE_LLM = False
client = None

try:
    import anthropic

    if "ANTHROPIC_API_KEY" in os.environ and os.environ["ANTHROPIC_API_KEY"] != "":
        client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        USE_LLM = True
        print("✅ Anthropic client initialized")
    else:
        USE_LLM = False
        print("⚠️ ANTHROPIC_API_KEY not found — using rule-based fallback")

except Exception as e:
    USE_LLM = False
    print("⚠️ LLM not available — using rule-based fallback")
    print("Reason:", e)


SYSTEM_PROMPT = """
You are a senior marketing risk analyst specializing in brand reputation.

For each social media post provided, respond ONLY with a valid JSON object.
Do not use markdown.
Do not add explanation outside JSON.

JSON format:
{
  "sentiment": "positive" | "neutral" | "negative",
  "risk_level": 0-10,
  "risk_category": "none" | "product" | "reputation" | "legal" | "viral" | "data_breach",
  "urgency": "low" | "medium" | "high" | "critical",
  "reasoning": "one short sentence"
}

Risk level guide:
0 = no risk
3 = minor concern
6 = significant risk
9-10 = brand crisis
"""


def safe_json_parse(text):
    """
    Safely parse JSON from Claude response.
    """

    try:
        return json.loads(text)

    except:
        start = text.find("{")
        end = text.rfind("}")

        if start != -1 and end != -1:
            json_text = text[start:end + 1]
            return json.loads(json_text)

        raise ValueError("Could not parse JSON response")


def analyze_post_llm(text):
    """
    Call Claude API to analyze a social post for brand risk.
    """

    response = client.messages.create(
        model="claude-3-5-sonnet-latest",
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": "Analyze this social media post: " + text
            }
        ]
    )

    raw_text = response.content[0].text
    result = safe_json_parse(raw_text)

    return result


def analyze_post_fallback(text):
    """
    Rule-based fallback scorer for demo without API key.
    """

    text_l = text.lower()

    crisis_kws = [
        "breach", "lawsuit", "recall", "trending", "leaked",
        "faked", "lied", "ftc", "investigation", "fabricated",
        "devastating", "change your passwords"
    ]

    legal_kws = [
        "lawsuit", "court", "legal", "ftc", "investigation",
        "misleading advertising"
    ]

    data_kws = [
        "data breach", "password", "passwords", "leak", "leaked"
    ]

    viral_kws = [
        "trending", "2m views", "influencer", "thread"
    ]

    moderate_kws = [
        "slipping", "worse", "broke", "overwhelmed", "wait",
        "cheap", "crashing", "confusing", "complaints"
    ]

    recovery_kws = [
        "apology", "refunds", "taking action", "hired",
        "improve", "support page", "acknowledges"
    ]

    positive_kws = [
        "smooth", "happy", "impressed", "slick", "discount",
        "good", "effective", "quality never drops", "well produced"
    ]

    if any(k in text_l for k in data_kws):
        return {
            "sentiment": "negative",
            "risk_level": 10,
            "risk_category": "data_breach",
            "urgency": "critical",
            "reasoning": "The post mentions a possible data breach or password risk."
        }

    elif any(k in text_l for k in legal_kws):
        return {
            "sentiment": "negative",
            "risk_level": 9,
            "risk_category": "legal",
            "urgency": "critical",
            "reasoning": "The post mentions legal or regulatory risk."
        }

    elif any(k in text_l for k in viral_kws):
        return {
            "sentiment": "negative",
            "risk_level": 8,
            "risk_category": "viral",
            "urgency": "high",
            "reasoning": "The post may spread quickly and damage brand reputation."
        }

    elif any(k in text_l for k in crisis_kws):
        return {
            "sentiment": "negative",
            "risk_level": 8,
            "risk_category": "reputation",
            "urgency": "high",
            "reasoning": "The post includes crisis-related reputation keywords."
        }

    elif any(k in text_l for k in moderate_kws):
        return {
            "sentiment": "negative",
            "risk_level": 5,
            "risk_category": "product",
            "urgency": "medium",
            "reasoning": "The post mentions product or service quality concerns."
        }

    elif any(k in text_l for k in recovery_kws):
        return {
            "sentiment": "neutral",
            "risk_level": 3,
            "risk_category": "reputation",
            "urgency": "medium",
            "reasoning": "The post shows recovery action but still relates to brand issues."
        }

    elif any(k in text_l for k in positive_kws):
        return {
            "sentiment": "positive",
            "risk_level": 0,
            "risk_category": "none",
            "urgency": "low",
            "reasoning": "The post has positive sentiment and low brand risk."
        }

    else:
        return {
            "sentiment": "neutral",
            "risk_level": 1,
            "risk_category": "none",
            "urgency": "low",
            "reasoning": "The post is neutral and has low risk."
        }


print("✅ Risk scoring functions are ready.")
print("USE_LLM:", USE_LLM)

⚠️ ANTHROPIC_API_KEY not found — using rule-based fallback
✅ Risk scoring functions are ready.
USE_LLM: False


In [17]:
# Module 4 — Score All Posts

results = []

if USE_LLM:
    analyze_fn = analyze_post_llm
else:
    analyze_fn = analyze_post_fallback

for index, row in posts_df.iterrows():

    try:
        result = analyze_fn(row["text"])

        result["post_id"] = row["id"]
        result["text"] = row["text"]
        result["true_type"] = row["type"]

        results.append(result)

    except Exception as e:
        print("Error on post", row["id"], ":", e)

        fallback_result = analyze_post_fallback(row["text"])
        fallback_result["post_id"] = row["id"]
        fallback_result["text"] = row["text"]
        fallback_result["true_type"] = row["type"]

        results.append(fallback_result)


results_df = pd.DataFrame(results)

# Reorder columns
results_df = results_df[
    [
        "post_id",
        "text",
        "true_type",
        "sentiment",
        "risk_level",
        "risk_category",
        "urgency",
        "reasoning"
    ]
]

print("✅ Scored posts:", len(results_df))

print("\nRisk level by true post type:")
display(results_df.groupby("true_type")["risk_level"].agg(["mean", "max", "count"]).round(2))

display(results_df.head())

✅ Scored posts: 30

Risk level by true post type:


,mean,max,count
true_type,,,
crisis,8.71,10,7
moderate,4.50,5,8
normal,0.20,1,10
recovery,3.40,5,5


,post_id,text,true_type,sentiment,risk_level,risk_category,urgency,reasoning
0,1,NovaByte ads are everywhere this week. Pretty ...,normal,positive,0,none,low,The post has positive sentiment and low brand ...
1,2,Just bought from NovaByte after seeing their a...,normal,positive,0,none,low,The post has positive sentiment and low brand ...
2,3,NovaByte radio ad got stuck in my head. Good j...,normal,positive,0,none,low,The post has positive sentiment and low brand ...
3,4,Saw NovaByte on TV last night. The new campaig...,normal,positive,0,none,low,The post has positive sentiment and low brand ...
4,5,"NovaByte discount code worked, 10% off my orde...",normal,positive,0,none,low,The post has positive sentiment and low brand ...


In [18]:
# Module 4 — Brand Risk Index Computation

def compute_bri(results_df):
    """
    Build weighted Brand Risk Index from 0 to 100.
    """

    urgency_weights = {
        "low": 1.0,
        "medium": 1.5,
        "high": 2.0,
        "critical": 3.0
    }

    category_penalty = {
        "none": 0,
        "product": 0.5,
        "reputation": 1,
        "viral": 2,
        "legal": 2,
        "data_breach": 3
    }

    df_bri = results_df.copy()

    df_bri["weight"] = df_bri["urgency"].map(urgency_weights).fillna(1.0)
    df_bri["penalty"] = df_bri["risk_category"].map(category_penalty).fillna(0)

    df_bri["adjusted_risk"] = df_bri["risk_level"] + df_bri["penalty"]
    df_bri["adjusted_risk"] = df_bri["adjusted_risk"].clip(0, 10)

    df_bri["weighted_risk"] = df_bri["adjusted_risk"] * df_bri["weight"]

    bri = (df_bri["weighted_risk"].sum() / (df_bri["weight"].sum() * 10)) * 100

    crisis_posts = (df_bri["risk_level"] >= 7).sum()
    high_urgency_posts = df_bri[df_bri["urgency"].isin(["high", "critical"])].shape[0]

    if bri < 30:
        status = "LOW"
    elif bri < 60:
        status = "MODERATE"
    else:
        status = "HIGH RISK"

    print("\n📊 Brand Risk Index BRI:", round(bri, 2), "/ 100")
    print("Status:", status)
    print("Posts analysed:", len(df_bri))
    print("Crisis posts:", crisis_posts)
    print("High or critical urgency posts:", high_urgency_posts)

    print("\n🚨 Top 5 Risk Posts:")

    top_risk = df_bri.nlargest(5, "adjusted_risk")[
        [
            "post_id",
            "text",
            "risk_level",
            "adjusted_risk",
            "risk_category",
            "urgency"
        ]
    ]

    display(top_risk)

    return bri, status, df_bri


bri_score, bri_status, scored_df = compute_bri(results_df)


📊 Brand Risk Index BRI: 56.74 / 100
Status: MODERATE
Posts analysed: 30
Crisis posts: 7
High or critical urgency posts: 7

🚨 Top 5 Risk Posts:


,post_id,text,risk_level,adjusted_risk,risk_category,urgency
18,19,BREAKING: Multiple users reporting NovaByte da...,10,10.00,data_breach,critical
20,21,Major influencer @TechReview just posted a dev...,8,10.00,viral,high
21,22,Lawsuit filed against NovaByte for misleading ...,10,10.00,data_breach,critical
22,23,#NovaByteLied is trending. Customers saying pr...,8,10.00,viral,high
23,24,Ex-NovaByte employee says the company faked en...,8,10.00,viral,high


In [19]:
# Module 4 — BRI Visualization

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Average Risk Level by Category",
        "Sentiment Distribution"
    ],
    specs=[
        [
            {"type": "bar"},
            {"type": "pie"}
        ]
    ]
)

cat_risk = scored_df.groupby("risk_category")["risk_level"].mean().sort_values(ascending=False)

fig.add_trace(
    go.Bar(
        x=cat_risk.index,
        y=cat_risk.values,
        name="Average Risk Level",
        text=[round(v, 1) for v in cat_risk.values],
        textposition="auto"
    ),
    row=1,
    col=1
)

sent_counts = scored_df["sentiment"].value_counts()

fig.add_trace(
    go.Pie(
        labels=sent_counts.index,
        values=sent_counts.values,
        hole=0.4,
        name="Sentiment"
    ),
    row=1,
    col=2
)

fig.update_layout(
    height=450,
    template="plotly_dark",
    title="Brand Risk Dashboard | BRI: " + str(round(bri_score, 1)) + "/100 | Status: " + bri_status,
    showlegend=True
)

fig.show()

In [20]:
# Module 4 — Additional Risk Visualizations

fig2 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Risk Level by Post",
        "Urgency Count"
    ]
)

fig2.add_trace(
    go.Bar(
        x=scored_df["post_id"],
        y=scored_df["risk_level"],
        name="Risk Level",
        text=scored_df["risk_level"],
        textposition="auto"
    ),
    row=1,
    col=1
)

urgency_counts = scored_df["urgency"].value_counts()

fig2.add_trace(
    go.Bar(
        x=urgency_counts.index,
        y=urgency_counts.values,
        name="Urgency Count",
        text=urgency_counts.values,
        textposition="auto"
    ),
    row=1,
    col=2
)

fig2.update_layout(
    height=450,
    template="plotly_dark",
    title="Social Media Risk Overview",
    showlegend=False
)

fig2.update_xaxes(title_text="Post ID", row=1, col=1)
fig2.update_yaxes(title_text="Risk Level", row=1, col=1)

fig2.update_xaxes(title_text="Urgency", row=1, col=2)
fig2.update_yaxes(title_text="Number of Posts", row=1, col=2)

fig2.show()

In [21]:
# Module 4 — Create Brand Risk Summary Table for Power BI export

brand_risk_summary_df = pd.DataFrame({
    "Metric": [
        "Brand Risk Index",
        "BRI Status",
        "Total Posts",
        "Average Risk Level",
        "Maximum Risk Level",
        "Crisis Posts",
        "High or Critical Urgency Posts",
        "Negative Sentiment Posts"
    ],
    "Value": [
        round(bri_score, 2),
        bri_status,
        len(scored_df),
        round(scored_df["risk_level"].mean(), 2),
        scored_df["risk_level"].max(),
        (scored_df["risk_level"] >= 7).sum(),
        scored_df[scored_df["urgency"].isin(["high", "critical"])].shape[0],
        (scored_df["sentiment"] == "negative").sum()
    ]
})

display(brand_risk_summary_df)
print("✅ Brand risk summary table is ready for export.")

,Metric,Value
0,Brand Risk Index,56.74
1,BRI Status,MODERATE
2,Total Posts,30
3,Average Risk Level,3.87
4,Maximum Risk Level,10
5,Crisis Posts,7
6,High or Critical Urgency Posts,7
7,Negative Sentiment Posts,15


✅ Brand risk summary table is ready for export.


## Module 4 Interpretation

In this module, I analyzed brand reputation risk using social media posts. The dataset contains synthetic posts about the brand NovaByte. Some posts are positive, some show moderate concerns, and some represent crisis situations such as data breach, legal issues, viral criticism, or false advertising claims.

The model or fallback logic assigned each post a sentiment, risk level, risk category, urgency, and short explanation. This helps the company quickly identify which posts need attention from the marketing or PR team.

I also calculated a Brand Risk Index, or BRI. This index summarizes the overall reputation risk from 0 to 100. Higher BRI means higher brand risk. Critical and high urgency posts receive more weight, and legal, viral, and data breach posts receive additional penalty points.

The dashboard shows average risk by category, sentiment distribution, risk level by post, and urgency counts. These visuals help managers understand the current reputation situation and decide what actions are needed.


---
##  Module 5 — Power BI Export & Dashboard Task

This module exports all analysis results to structured Excel files optimized for Power BI import.

### What you'll build in Power BI:
1. **Campaign Performance Overview** — KPI cards, spend vs. sales trend line
2. **Risk Heatmap** — ROI volatility by channel and time period  
3. **Anomaly Timeline** — Highlighted anomaly weeks with drill-through
4. **Brand Risk Gauge** — BRI score with traffic light conditional formatting
5. **Monte Carlo Summary** — VaR/CVaR summary table with risk band slicers

**Step 1:** Run the cell below to export all sheets  
**Step 2:** Open Power BI Desktop → Get Data → Excel  
**Step 3:** Follow the Power BI task guide at the bottom of this notebook

In [22]:
# Module 5 — Export All Data to Power BI-Ready Excel

import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
import os

EXPORT_PATH = "marketing_risk_powerbi.xlsx"

# ------------------------------------------------------------
# Sheet 1: Campaign Performance
# ------------------------------------------------------------

campaign_cols = [
    "Date",
    "Total_Spend",
    "TV",
    "Radio",
    "Newspaper",
    "Sales",
    "ROI",
    "CPR",
    "TV_Share",
    "Radio_Share",
    "Newspaper_Share",
    "Low_ROI_Flag",
    "ROI_4w_avg",
    "Spend_4w_avg",
    "Sales_4w_avg",
    "Campaign_Risk_Level"
]

available_campaign_cols = []

for col in campaign_cols:
    if col in df.columns:
        available_campaign_cols.append(col)

df_export = df[available_campaign_cols].copy()

rename_map = {
    "TV": "TV_Spend",
    "Radio": "Radio_Spend",
    "Newspaper": "Newspaper_Spend",
    "ROI": "ROI_Pct",
    "TV_Share": "TV_Share_Pct",
    "Radio_Share": "Radio_Share_Pct",
    "Newspaper_Share": "Newspaper_Share_Pct",
    "ROI_4w_avg": "ROI_4W_Rolling",
    "Spend_4w_avg": "Spend_4W_Rolling",
    "Sales_4w_avg": "Sales_4W_Rolling"
}

df_export = df_export.rename(columns=rename_map)

# ------------------------------------------------------------
# Sheet 2: Anomaly Results
# ------------------------------------------------------------

anomaly_cols = [
    "Date",
    "Total_Spend",
    "Sales",
    "ROI",
    "CPR",
    "anomaly_score",
    "is_anomaly",
    "Anomaly_Status",
    "Spend_pct_change",
    "Sales_pct_change",
    "ROI_pct_change",
    "TV_Radio_Ratio_Change",
    "Efficiency",
    "Efficiency_Change",
    "Max_Channel_Share"
]

available_anomaly_cols = []

for col in anomaly_cols:
    if col in df_feat.columns:
        available_anomaly_cols.append(col)

df_anomaly_export = df_feat[available_anomaly_cols].copy()

if "ROI" in df_anomaly_export.columns:
    df_anomaly_export = df_anomaly_export.rename(columns={"ROI": "ROI_Pct"})

if "is_anomaly" in df_anomaly_export.columns:
    df_anomaly_export["Risk_Level"] = np.where(
        df_anomaly_export["is_anomaly"] == True,
        "HIGH",
        "NORMAL"
    )

# ------------------------------------------------------------
# Sheet 3: Monte Carlo Simulation
# ------------------------------------------------------------

if "simulation_df" in globals():
    monte_carlo_simulation = simulation_df.copy()
else:
    monte_carlo_simulation = pd.DataFrame({
        "simulation_id": range(1, N_SIM + 1),
        "simulated_spend": spend_sim,
        "simulated_revenue": revenue_sim,
        "simulated_roi": roi_sim
    })

    def simulation_risk_level_export(roi):
        if roi <= metrics["var_5"]:
            return "High Risk"
        elif roi <= metrics["mean_roi"]:
            return "Medium Risk"
        else:
            return "Low Risk"

    monte_carlo_simulation["risk_level"] = monte_carlo_simulation["simulated_roi"].apply(simulation_risk_level_export)

# ------------------------------------------------------------
# Sheet 4: Monte Carlo Summary
# ------------------------------------------------------------

mc_summary = pd.DataFrame({
    "Metric": [
        "Mean ROI",
        "Median ROI",
        "ROI Standard Deviation",
        "VaR 5%",
        "VaR 10%",
        "CVaR",
        "Probability of Loss",
        "Best Case ROI",
        "Worst Case ROI"
    ],
    "Value_Pct": [
        metrics["mean_roi"],
        metrics["median_roi"],
        metrics["std_roi"],
        metrics["var_5"],
        metrics["var_10"],
        metrics["cvar"],
        metrics["prob_loss"],
        metrics["best_case"],
        metrics["worst_case"]
    ],
    "Category": [
        "Performance",
        "Performance",
        "Risk",
        "Risk",
        "Risk",
        "Risk",
        "Risk",
        "Opportunity",
        "Risk"
    ]
})

# ------------------------------------------------------------
# Sheet 5: Brand Risk Scores
# ------------------------------------------------------------

brand_cols = [
    "post_id",
    "text",
    "true_type",
    "sentiment",
    "risk_level",
    "risk_category",
    "urgency",
    "reasoning",
    "adjusted_risk",
    "weighted_risk"
]

available_brand_cols = []

for col in brand_cols:
    if col in scored_df.columns:
        available_brand_cols.append(col)

brand_export = scored_df[available_brand_cols].copy()
brand_export["BRI_Score"] = bri_score
brand_export["BRI_Status"] = bri_status

# ------------------------------------------------------------
# Sheet 6: Brand Risk Summary
# ------------------------------------------------------------

if "brand_risk_summary_df" in globals():
    brand_summary_export = brand_risk_summary_df.copy()
else:
    brand_summary_export = pd.DataFrame({
        "Metric": [
            "Brand Risk Index",
            "BRI Status",
            "Total Posts",
            "Average Risk Level",
            "Maximum Risk Level",
            "Crisis Posts",
            "High or Critical Urgency Posts",
            "Negative Sentiment Posts"
        ],
        "Value": [
            round(bri_score, 2),
            bri_status,
            len(scored_df),
            round(scored_df["risk_level"].mean(), 2),
            scored_df["risk_level"].max(),
            (scored_df["risk_level"] >= 7).sum(),
            scored_df[scored_df["urgency"].isin(["high", "critical"])].shape[0],
            (scored_df["sentiment"] == "negative").sum()
        ]
    })

# ------------------------------------------------------------
# Sheet 7: KPI Summary
# ------------------------------------------------------------

best_channel = "TV"

avg_tv_roi = df[df["TV_Share"] == df[["TV_Share", "Radio_Share", "Newspaper_Share"]].max(axis=1)]["ROI"].mean()
avg_radio_roi = df[df["Radio_Share"] == df[["TV_Share", "Radio_Share", "Newspaper_Share"]].max(axis=1)]["ROI"].mean()
avg_newspaper_roi = df[df["Newspaper_Share"] == df[["TV_Share", "Radio_Share", "Newspaper_Share"]].max(axis=1)]["ROI"].mean()

channel_roi_dict = {
    "TV": avg_tv_roi,
    "Radio": avg_radio_roi,
    "Newspaper": avg_newspaper_roi
}

best_channel = max(channel_roi_dict, key=channel_roi_dict.get)
best_channel_roi = channel_roi_dict[best_channel]

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Campaigns Analysed",
        "Average ROI",
        "Average Sales",
        "Average Total Spend",
        "VaR 5%",
        "CVaR",
        "Probability of Loss",
        "Anomalies Detected",
        "Brand Risk Index",
        "Low ROI Weeks",
        "Best Channel",
        "Best Channel ROI"
    ],
    "Value": [
        len(df),
        round(df["ROI"].mean(), 2),
        round(df["Sales"].mean(), 2),
        round(df["Total_Spend"].mean(), 2),
        round(metrics["var_5"], 2),
        round(metrics["cvar"], 2),
        round(metrics["prob_loss"], 2),
        int(df_feat["is_anomaly"].sum()),
        round(bri_score, 2),
        int(df["Low_ROI_Flag"].sum()),
        best_channel,
        round(best_channel_roi, 2)
    ],
    "Unit": [
        "weeks",
        "%",
        "sales units",
        "spend units",
        "%",
        "%",
        "%",
        "weeks",
        "/100",
        "weeks",
        "channel",
        "%"
    ],
    "Status": [
        "INFO",
        "GOOD" if df["ROI"].mean() > -50 else "WARN",
        "INFO",
        "INFO",
        "RISK" if metrics["var_5"] < -50 else "WARN",
        "RISK" if metrics["cvar"] < -50 else "WARN",
        "RISK" if metrics["prob_loss"] > 50 else "WARN",
        "RISK" if df_feat["is_anomaly"].sum() > 10 else "WARN",
        "RISK" if bri_score > 60 else "WARN" if bri_score > 30 else "GOOD",
        "WARN",
        "GOOD",
        "GOOD"
    ]
})

# ------------------------------------------------------------
# Write Excel
# ------------------------------------------------------------

with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
    df_export.to_excel(writer, sheet_name="Campaign_Performance", index=False)
    df_anomaly_export.to_excel(writer, sheet_name="Anomaly_Results", index=False)
    monte_carlo_simulation.to_excel(writer, sheet_name="MonteCarlo_Simulation", index=False)
    mc_summary.to_excel(writer, sheet_name="MonteCarlo_Summary", index=False)
    brand_export.to_excel(writer, sheet_name="Brand_Risk_Scores", index=False)
    brand_summary_export.to_excel(writer, sheet_name="Brand_Risk_Summary", index=False)
    kpi_summary.to_excel(writer, sheet_name="KPI_Summary", index=False)

# ------------------------------------------------------------
# Excel Styling
# ------------------------------------------------------------

wb = openpyxl.load_workbook(EXPORT_PATH)

header_fill = PatternFill(start_color="1A1A26", end_color="1A1A26", fill_type="solid")
header_font = Font(color="00E5C0", bold=True, name="Calibri", size=11)

risk_fill = PatternFill(start_color="FF4F5E", end_color="FF4F5E", fill_type="solid")
warn_fill = PatternFill(start_color="FFB347", end_color="FFB347", fill_type="solid")
good_fill = PatternFill(start_color="00E5C0", end_color="00E5C0", fill_type="solid")

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center")

    ws.freeze_panes = "A2"

    for col in ws.columns:
        max_len = 0

        for cell in col:
            value = str(cell.value) if cell.value is not None else ""
            if len(value) > max_len:
                max_len = len(value)

        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 45)

# Status color in KPI Summary
ws_kpi = wb["KPI_Summary"]

for row in ws_kpi.iter_rows(min_row=2):
    status_cell = row[3]

    if status_cell.value == "RISK":
        status_cell.fill = risk_fill
        status_cell.font = Font(bold=True, color="FFFFFF")
    elif status_cell.value == "WARN":
        status_cell.fill = warn_fill
        status_cell.font = Font(bold=True)
    elif status_cell.value == "GOOD":
        status_cell.fill = good_fill
        status_cell.font = Font(bold=True)

wb.save(EXPORT_PATH)

print("✅ Exported to:", EXPORT_PATH)
print("Sheets:")
print("- Campaign_Performance")
print("- Anomaly_Results")
print("- MonteCarlo_Simulation")
print("- MonteCarlo_Summary")
print("- Brand_Risk_Scores")
print("- Brand_Risk_Summary")
print("- KPI_Summary")
print()
print("File size:", round(os.path.getsize(EXPORT_PATH) / 1024, 1), "KB")

✅ Exported to: marketing_risk_powerbi.xlsx
Sheets:
- Campaign_Performance
- Anomaly_Results
- MonteCarlo_Simulation
- MonteCarlo_Summary
- Brand_Risk_Scores
- Brand_Risk_Summary
- KPI_Summary

File size: 673.7 KB


---
##  Power BI Task Guide

### Setup (5 min)
1. Open **Power BI Desktop** (free download from Microsoft)
2. Click **Home → Get Data → Excel Workbook**
3. Select `marketing_risk_powerbi.xlsx` → load all 5 sheets
4. In Power Query Editor: confirm data types (dates as Date, numerics as Decimal)

---

### Task A — Campaign Performance Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Line Chart | X=Date, Y=Sales, Y2=Total_Spend | Dual axis, 4-week rolling average |
| KPI Card | Value=Avg ROI_Pct | Conditional format: Green>20%, Red<0% |
| Stacked Bar | X=Date, Values=TV_Spend, Radio_Spend, Newspaper_Spend | Channel breakdown |
| Scatter Plot | X=Total_Spend, Y=Sales, Size=ROI_Pct | Bubble chart with Low_ROI_Flag colour |

### Task B — Risk Dashboard Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Gauge | Value=BRI_Score | Min=0, Max=100, Target=30 |
| Table | All columns from KPI_Summary | Conditional format Status column |
| Clustered Bar | X=Metric, Y=Value_Pct from MonteCarlo | Highlight VaR/CVaR bars red |
| Card (3x) | VaR 5%, CVaR, P(Loss) | Font colour = red |

### Task C — Anomaly Monitor Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Line + Scatter | X=Index, Y=Sales, Scatter=is_anomaly | Mark anomaly points red ✕ |
| Bar Chart | X=Index, Y=anomaly_score | Colour by Risk_Level column |
| Slicer | Risk_Level | Dropdown filter: HIGH / NORMAL |
| Drill-through | Click anomaly row → shows post detail card | Set up drill-through filter |

### Task D — Brand Reputation Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Donut Chart | Legend=sentiment, Values=count | Colours: green/grey/red |
| Treemap | Category=risk_category, Size=risk_level | Colour saturation = urgency |
| Table | text, risk_level, urgency, reasoning | Sort by risk_level desc, top 10 |
| Gauge | BRI_Score | Zone bands: 0-30 green, 30-60 amber, 60-100 red |

### Task E — Extension: DAX Measures *(bonus)*
```dax
// Average ROI excluding anomaly weeks
Clean_Avg_ROI =
    CALCULATE(AVERAGE(Campaign_Performance[ROI_Pct]),
              Anomaly_Results[Risk_Level] = "NORMAL")

// Brand Risk Traffic Light
BRI_Status =
    SWITCH(TRUE(),
        [BRI_Score] < 30, "🟢 LOW",
        [BRI_Score] < 60, "🟡 MODERATE",
        "🔴 HIGH RISK")

// Rolling 4-week ROI
ROI_4W =
    AVERAGEX(
        DATESINPERIOD(Campaign_Performance[Date], LASTDATE(Campaign_Performance[Date]), -28, DAY),
        Campaign_Performance[ROI_Pct])
```

---

### Deliverables
- [ ] `.pbix` file with all 4 pages completed
- [ ] At least 1 custom DAX measure per page
- [ ] Cross-page drill-through working (click anomaly → see brand risk for that period)
- [ ] Published to Power BI Service (free account) with shareable link
- [ ] 2-minute screen recording walkthrough as CMO presenting to the board

---
##  Lab Summary & Reflection Questions

Run the cell below to print your final risk summary, then answer the reflection questions.


In [24]:
print("=" * 70)
print("  MARKETING RISK & RESILIENCE LAB — FINAL SUMMARY")
print("=" * 70)

print()
print("  Dataset:                 ", len(df), "weeks of advertising performance data")
print("  Mean ROI:                ", round(df["ROI"].mean(), 2), "%")
print("  ROI Std Dev:             ", round(df["ROI"].std(), 2), "%")
print("  Average Sales:           ", round(df["Sales"].mean(), 2))
print("  Average Total Spend:     ", round(df["Total_Spend"].mean(), 2))

print()
print("  Monte Carlo VaR 5%:      ", round(metrics["var_5"], 2), "%")
print("  Monte Carlo CVaR:        ", round(metrics["cvar"], 2), "%")
print("  Probability of Loss:     ", round(metrics["prob_loss"], 2), "%")

print()
print("  Anomalies Detected:      ", int(df_feat["is_anomaly"].sum()), "weeks flagged by Isolation Forest")
print("  Low ROI Weeks:           ", int(df["Low_ROI_Flag"].sum()))

print()
print("  Brand Risk Index:        ", round(bri_score, 2), "/100")
print("  Brand Risk Status:       ", bri_status)

print()
if bri_score < 30:
    overall_status = "LOW RISK"
elif bri_score < 60:
    overall_status = "MODERATE RISK"
else:
    overall_status = "HIGH RISK"

print("  Overall Brand Status:    ", overall_status)

print()
print("  Power BI Export:         marketing_risk_powerbi.xlsx")
print("=" * 70)

print()
print("Reflection Questions:")
print("1. Which distribution did you choose for spend and why?")
print("2. How would correlated risk factors change your VaR estimate?")
print("3. What false positives did the anomaly model produce? Why?")
print("4. How reliable is LLM risk scoring vs rule-based? Where does it fail?")
print("5. What additional data would improve the Brand Risk Index?")

  MARKETING RISK & RESILIENCE LAB — FINAL SUMMARY

  Dataset:                  400 weeks of advertising performance data
  Mean ROI:                 -93.02 %
  ROI Std Dev:              1.62 %
  Average Sales:            15.04
  Average Total Spend:      220.0

  Monte Carlo VaR 5%:       -95.64 %
  Monte Carlo CVaR:         -96.14 %
  Probability of Loss:      100.0 %

  Anomalies Detected:       20 weeks flagged by Isolation Forest
  Low ROI Weeks:            40

  Brand Risk Index:         56.74 /100
  Brand Risk Status:        MODERATE

  Overall Brand Status:     MODERATE RISK

  Power BI Export:         marketing_risk_powerbi.xlsx

Reflection Questions:
1. Which distribution did you choose for spend and why?
2. How would correlated risk factors change your VaR estimate?
3. What false positives did the anomaly model produce? Why?
4. How reliable is LLM risk scoring vs rule-based? Where does it fail?
5. What additional data would improve the Brand Risk Index?


## Reflection Questions

**1. Which distribution did you choose for spend and why?**
I chose a **lognormal distribution** because advertising spend is always positive and usually right-skewed. Most campaigns have normal budgets, but some campaigns have very high spending.

**2. How would correlated risk factors change your VaR estimate?**
Correlated risk factors would likely increase VaR. For example, if high spend happens together with poor market conditions, the worst-case scenarios become more serious.

**3. What false positives did the anomaly model produce? Why?**
The model may flag successful high-budget campaigns as anomalies. This happens because Isolation Forest detects unusual patterns, not only bad performance.

**4. How reliable is LLM risk scoring vs rule-based? Where does it fail?**
LLM scoring is more flexible because it understands context and tone. However, it can fail with sarcasm, slang, or unclear posts. Rule-based scoring is easier to explain but can miss context.

**5. What additional data would improve the Brand Risk Index?**
BRI would improve with likes, shares, comments, views, author follower count, platform type, post time, and real customer complaint data.
